# Getting Started with Automated-LLM-Probes

In [1]:
from __future__ import annotations
import os,time,tqdm,pandas as pd
from pathlib import Path
import automated_intelligence_tests as ait
import automated_llm_probes as alp
assert os.environ.get("OPENAI_API_KEY")

### Models available

In [2]:
models = alp.load_models()
print(f"|Models available: {len(models)}|\n")
for m in models:
    print(f"  {m['name'][:20]:20s} {m['vendor'][:11]:12s} \
{m['api'][:11]:12s} {m['model_id'][:12]:14s} {m['status'][:12]:12s}")

|Models available: 91|

  grok-code-fast       xai          spacexai     grok-code-fa   ok          
  grok-2               xai          spacexai     grok-2         failed      
  grok-4               xai          spacexai     grok-4         ok          
  grok-4.2             xai          spacexai     grok-4.20-03   ok          
  grok-4.2-reasoning   xai          spacexai     grok-4.20-03   ok          
  grok-4.3             xai          spacexai     grok-4.3       ok          
  grok-4.5             xai          spacexai     grok-4.5       ok          
  grok-build-0.1       xai          spacexai     grok-build-0   ok          
  grok-4.6             xai          spacexai     grok-4.6       ok          
  grok-4.2-multi-agent xai          spacexai     grok-4.20-mu   failed      
  hunyuan-3            tencent      hunyuan      hy3            ok          
  hunyuan-lite         tencent      hunyuan      hunyuan-lite   failed      
  qwen-turbo           qwen         qwen         qwe

### Probe models

In [3]:
def probe_models(model_rows,prompt=[{"role":"user","content":"reply ok"}]):
    retries = alp.MAX_RETRIES; alp.MAX_RETRIES = 1
    try:
        for model in tqdm.tqdm(model_rows):
            start = time.time()
            try:
                model['reply'] = alp.call_model(model,prompt)
                model["status"] = "ok" if model['reply'].strip() else "failed"
                model["errors"] = None
            except Exception as e:
                model['reply'] = None
                model["status"] = 'failed'
                model["errors"] = str(e).lower()
            model["speed_per_call"] = f"{round((time.time()-start),2)} seconds"
    finally:
        alp.MAX_RETRIES = retries
    return model_rows

models = alp.load_models()
probes = probe_models(models)
probes = pd.DataFrame(probes).set_index('name')

100%|████████████████████████████████████████████████████████████████████| 91/91 [04:51<00:00,  3.21s/it]


In [4]:
probes.to_csv('./models.csv')
probes.head()

,vendor,api,model_id,release_date,temperature,speed_per_call,status,reply,errors
name,,,,,,,,,
grok-code-fast,xai,spacexai,grok-code-fast,26-Aug-25,0.5,9.83 seconds,ok,Ok.,None
grok-2,xai,spacexai,grok-2,13-Aug-24,default,2.19 seconds,failed,None,grok-2 failed after 1 tries: error code: 400 -...
grok-4,xai,spacexai,grok-4,9-Jul-25,0.5,1.95 seconds,ok,ok,None
grok-4.2,xai,spacexai,grok-4.20-0309-non-reasoning,9-Mar-26,0.5,0.69 seconds,ok,ok,None
grok-4.2-reasoning,xai,spacexai,grok-4.20-0309-reasoning,9-Mar-26,0.5,1.95 seconds,ok,ok,None


### Probed tasks

In [5]:
path = Path('./data/'); n=1
print(f"|Probed tasks: {len([p for p in path.iterdir() if p.is_dir()])}|\n")
for d in path.iterdir():
    if (d.is_dir()) and ('.' not in d.name):
        count = sum(1 for f in d.rglob('*') if f.is_file())
        print(f"  {n}. {d.name.upper()[:7]:8}:  {count}"); n+=1

|Probed tasks: 4|

  1. AUT     :  16574
  2. CWT     :  22194
  3. DAT     :  10781


### Tests availabe

In [6]:
ait_counts = ait.list_available_tests()
print(f'|Available tests: {len(ait_counts)}|\n')
for i,v in ait_counts.items():
    print(f'  {i} : {v}')

|Available tests: 4|

  AUT : Alternative Uses Task
  CAT : Convergent Association Task
  CWT : Creative Writing Task
  DAT : Divergent Association Task


### Trial-run

In [7]:
models_to_try = ['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

cues_to_try = ["brick", "paperclip"]

models = alp.ready_models()
models_to_try = [m for m in models if m["name"] in models_to_try]
sorted([m["name"] for m in models_to_try])

['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

In [8]:
alp.collect("AUT", models=models_to_try, n_per_model=0)

  grok-4.2: 832/0 done — skip
  grok-4.3: 825/0 done — skip
  grok-4.5: 828/0 done — skip
  grok-build-0.1: 620/0 done — skip
  gpt-3.5-turbo: 921/0 done — skip
  gpt-4-turbo: 825/0 done — skip
  gpt-4o: 870/0 done — skip
  gpt-5.4: 861/0 done — skip
  gpt-4o-mini: 829/0 done — skip
  gpt-4-turbo: 825/0 done — skip
  llama-4-guard-12b: 600/0 done — skip
  llama-4-scout: 620/0 done — skip
  llama-4-maverick: 600/0 done — skip
  llama-3.2-3b: 600/0 done — skip
  llama-3.1-8b: 600/0 done — skip
  claude-sonnet-4.5: 827/0 done — skip
  claude-haiku-4.5: 830/0 done — skip
  claude-opus-4.5: 829/0 done — skip
  claude-opus-4.7: 859/0 done — skip
  claude-opus-5: 855/0 done — skip


### Load functions

In [16]:
...
...

### Parse & merge data

In [14]:
for task in (
    "dat", 
    "aut",
    "cwt"):
    
    print(f"Parsing {task.upper()}...")
    task = task.lower()
    df = pd.DataFrame.from_dict(alp.parse_and_merge(task),orient='index')
    df = load_task(df,task)
    print(df.shape,df.columns)
    df.to_csv(f"./data/{task.upper()}_AI_2026.csv", index=False)
    print('Saved...')

Parsing DAT...


dat: 100%|████████████████████████████████████████████████████████| 10778/10778 [00:18<00:00, 581.23it/s]


(10643, 21) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'noun_0', 'noun_1', 'noun_2', 'noun_3', 'noun_4', 'noun_5',
       'noun_6', 'noun_7', 'noun_8', 'noun_9', 'prompt', 'response_clean',
       'ts_utc', 'hash'],
      dtype='object')
Saved...
Parsing AUT...


aut: 100%|████████████████████████████████████████████████████████| 17210/17210 [00:44<00:00, 389.29it/s]


(17188, 11) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'prompt', 'response_clean', 'ts_utc', 'hash'],
      dtype='object')
Saved...
Parsing CWT...


cwt: 100%|████████████████████████████████████████████████████████| 23857/23857 [01:04<00:00, 369.03it/s]


(23750, 11) Index(['task', 'model_name', 'model_id', 'provider', 'rep', 'temperature',
       'cue', 'prompt', 'response_clean', 'ts_utc', 'hash'],
      dtype='object')
Saved...


In [13]:
raw = pd.DataFrame.from_dict(alp.parse_and_merge("dat"), orient="index")
print(raw.model_name.value_counts())
print(raw.loc[raw.model_name.eq(m),"temperature"].fillna("default").value_counts())

dat: 100%|████████████████████████████████████████████████████████| 10778/10778 [00:23<00:00, 466.10it/s]


model_name
claude-opus-4.7      700
gpt-3.5-turbo        600
grok-4.3             600
grok-build-0.1       600
grok-4.5             600
llama-4-maverick     450
claude-opus-5        450
llama-4-scout        450
gpt-4-turbo          450
gpt-4o-mini          450
llama-3.1-8b         450
llama-3.2-3b         450
llama-4-guard-12b    450
gpt-4o               450
claude-haiku-4.5     450
gpt-5.4              450
claude-sonnet-4.5    450
claude-opus-4.5      450
grok-4.2             400
grok-4.6             350
gpt-5.6-sol          350
deepseek-3.2         275
deepseek-2.5-chat    275
kimi-k2               20
deepseek-r1            5
grok-code-fast         5
gpt-5-mini             5
gpt-5                  5
qwen-turbo             3
Name: count, dtype: int64
Series([], Name: count, dtype: int64)
